# Day 5: Session 5A - The Split-Apply-Combine Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/5a_grouping_data.html)

Date: 09/04/2026

In [45]:
import pandas as pd
url = 'https://eds-217-essential-python.github.io/data/messy_field_survey.csv'
survey = pd.read_csv(url)

survey = survey.drop_duplicates()

survey["site"] = survey["site"].str.strip().str.lower().str.replace("-", "_")
# survey['site'] = survey['site'].str.lower()
# survey['site'] = survey['site'].str.replace('-', '_')

survey["pH"] = survey["pH"].str.replace(",", ".").astype(float)
# survey['pH'] = survey['pH'].astype(float)

survey = survey.dropna(
    subset=["temperature_c", "dissolved_oxygen_mg_L", "conductivity_uS_cm"]
)
survey["n_replicates"] = survey["n_replicates"].fillna(1).astype(int)
#survey["n_replicates"] = survey["n_replicates"].astype(int)

survey = survey[survey["temperature_c"] > -100].copy()
survey = survey.rename(columns={"collection date": "collection_date"})

survey.shape


(255, 7)

In [46]:
survey.head()

,site,collection_date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates
0,site_f,2025-07-28,24.1,5.89,5.51,899.1,4
1,site_d,2025-08-05,12.5,7.74,9.97,300.1,4
2,site_c,2025-07-04,22.6,6.42,7.19,815.8,4
3,site_d,2025-08-15,16.0,7.74,9.73,250.7,3
4,site_a,2025-06-24,14.7,7.64,9.19,338.8,3


In [47]:
survey['dissolved_oxygen_mg_L'].mean()

8.062039215686275

In [48]:
site_d = survey[survey['site'] == 'site_d']
print(site_d['dissolved_oxygen_mg_L'].mean())

site_f = survey[survey['site'] == 'site_f']
print(site_f['dissolved_oxygen_mg_L'].mean())

10.053333333333333
6.180232558139535


In [49]:
# always need to specify which column you want to group by
grouped = survey.groupby('site')

In [50]:
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

In [51]:
# this is the mean of the dissolved oxygen at each site in one step. since we used the groupby function
result = grouped['dissolved_oxygen_mg_L'].mean()

In [52]:
# it just made a series
print(result)
print(type(result))

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
<class 'pandas.core.series.Series'>


In [53]:
# now i can find the specific result for a site by calling the column.
result['site_a']

9.485909090909091

In [54]:
result.idxmax()

'site_d'

In [55]:
result.idxmin()

'site_f'

In [56]:
# step 1 - make the groupby object
grouped = survey.groupby('site')

# step 2 - aggregate on the object
grouped['dissolved_oxygen_mg_L'].mean()

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64

In [57]:
# this is the best way to do it. the most efficient. 

survey.groupby('site')['dissolved_oxygen_mg_L'].mean()

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64

In [58]:
# this is the format for this

# df.groupby('key')['column'].aggregation()
#          split      pick        apply

In [59]:
# groupby('key') says which piles to make. The key is the column whose values name the groups.
# ['column'] says which column to do analysis on.
# .mean() says what arithmetic.

In [60]:
mean_do = survey.groupby('site')['dissolved_oxygen_mg_L'].mean()

print(type(mean_do))
print(mean_do.index)

<class 'pandas.core.series.Series'>
Index(['site_a', 'site_b', 'site_c', 'site_d', 'site_e', 'site_f'], dtype='object', name='site')


In [61]:
mean_do['site_c']

6.873111111111111

In [62]:
mean_do.idxmax()

'site_d'

In [63]:
mean_do.idxmin()

'site_f'

In [64]:
result = survey.groupby('site')['temperature_c'].mean()
result.idxmax()

'site_f'

In [65]:
survey.groupby('site')['dissolved_oxygen_mg_L'].max()

site
site_a    11.12
site_b     9.86
site_c     7.99
site_d    11.43
site_e     8.79
site_f     7.77
Name: dissolved_oxygen_mg_L, dtype: float64

In [66]:
survey.groupby('site')['dissolved_oxygen_mg_L'].min()

site
site_a    8.21
site_b    7.33
site_c    5.65
site_d    8.60
site_e    5.16
site_f    4.14
Name: dissolved_oxygen_mg_L, dtype: float64

In [67]:
grouped['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

In [68]:
# seeing how big the sample size is
survey.groupby('site')['dissolved_oxygen_mg_L'].count()

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64

In [69]:
# this is for the whole column

survey['dissolved_oxygen_mg_L'].count()

255

In [70]:
survey.groupby('site')['pH'].max()


site
site_a    7.92
site_b    7.41
site_c    7.10
site_d    8.33
site_e    7.47
site_f    6.98
Name: pH, dtype: float64

In [71]:
survey.groupby('site')['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

What is the highest pH recorded at each site?
- site_a

How many bottles were filled in total at each site? (n_replicates counts bottles.)
- look above


In [72]:
def classify_ph(value):
    """Label a pH value as acidic, neutral, or alkaline."""
    if value < 6.5:
        return 'acidic'
    elif value > 7.5:
        return 'alkaline'
    else:
        return 'neutral'


survey['ph_class'] = survey['pH'].apply(classify_ph)
survey['ph_class'].value_counts()

ph_class
neutral     171
acidic       42
alkaline     42
Name: count, dtype: int64

In [73]:
survey.groupby('ph_class')['dissolved_oxygen_mg_L'].mean()

ph_class
acidic      6.350000
alkaline    9.948571
neutral     8.019181
Name: dissolved_oxygen_mg_L, dtype: float64

In [74]:
# group by takes data and puts it bins.

In [75]:
print(survey['site'].value_counts())
print(survey.groupby('site')['site'].count())

site
site_c    45
site_a    44
site_e    43
site_f    43
site_b    41
site_d    39
Name: count, dtype: int64
site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: site, dtype: int64


In [76]:
print(survey.groupby('site')['dissolved_oxygen_mg_L'].count())
print(survey.groupby('site')['dissolved_oxygen_mg_L'].mean())
print(survey.groupby('site')['temperature_c'].mean())

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64
site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
site
site_a    16.600000
site_b    18.102439
site_c    21.911111
site_d    15.179487
site_e    19.730233
site_f    23.297674
Name: temperature_c, dtype: float64


# Day 5: Session 5B - Several Answers at Once

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/5b_aggregating_data.html)

Date: 09/04/2026

pass a list to .agg() to get several summaries of one column

pass a dictionary to .agg() to summarise several columns differently

apply the top-N pattern to a grouped result

recognize a MultiIndex produced by grouping on two keys, and flatten it with .reset_index()

explain why a grouped mean should always be reported with the count it was computed from

In [77]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/national_parks.csv'
parks = pd.read_csv(url)

# Remove the rows that have park Totals in them and make a new dataframe:
by_year = parks[~(parks['year'] == 'Total')]
national_parks = by_year[by_year['unit_type'] == 'National Park'].copy()

national_parks.shape

(4682, 12)

In [78]:
national_parks.head()

,year,gnis_id,geometry,metadata,number_of_records,parkname,region,state,unit_code,unit_name,unit_type,visitors
0,1904,1163670,POLYGON,NaN,1,Crater Lake,PW,OR,CRLA,Crater Lake National Park,National Park,1500.0
3,1935,1530459,MULTIPOLYGON,NaN,1,Olympic,PW,WA,OLYM,Olympic National Park,National Park,2200.0
5,1919,578853,MULTIPOLYGON,NaN,1,NaN,NE,ME,ACAD,Acadia National Park,National Park,64000.0
8,1944,1377082,POLYGON,NaN,1,NaN,IM,TX,BIBE,Big Bend National Park,National Park,1409.0
22,1948,293666,POLYGON,NaN,1,NaN,SE,FL,EVER,Everglades National Park,National Park,7482.0


In [79]:
national_parks.groupby('region')['visitors'].mean()

region
AK    1.204881e+05
IM    7.931557e+05
MW    6.108661e+05
NC    5.107866e+05
NE    1.618013e+06
PW    6.627740e+05
SE    1.572087e+06
Name: visitors, dtype: float64

In [80]:
national_parks.groupby('region')['visitors'].count()

region
AK     488
IM    1590
MW     533
NC      46
NE     179
PW    1393
SE     453
Name: visitors, dtype: int64

In [81]:
# .agg takes a list of commands - shown below, pay attention to the syntax... its weird
result = national_parks.groupby('region')['visitors'].agg(['count', 'mean', 'max'])

In [82]:
# its a dataframe

type(result)

pandas.core.frame.DataFrame

agg(['mean']) with one name in the list gives you a DataFrame with one column. .mean() gives you a Series. Same numbers, in a different kind of object. Ask for the one you want to work with next.

In [83]:
national_parks.groupby('region')['unit_name'].nunique()

region
AK     9
IM    18
MW     7
NC     1
NE     2
PW    17
SE     7
Name: unit_name, dtype: int64

In [84]:
# you can also save all of the functions into an object and then call that as an argument for the .agg([]) function. 
national_parks.groupby('state')['visitors'].agg(['mean', 'max', 'count'])

,mean,max,count
state,,,
AK,1.204881e+05,592431.0,488
AR,8.443021e+05,2092400.0,109
AS,8.745692e+03,28892.0,13
AZ,1.009639e+06,5969811.0,290
CA,6.076323e+05,5028868.0,789
CO,6.353273e+05,4517585.0,378
FL,3.961249e+05,1534328.0,194
HI,8.838688e+05,2247974.0,153
KY,1.046299e+06,2396234.0,81


In [ ]:
# on the left is the column and on the right it is the command that you want to do to that column.
# for multiple columns we do not select a single column
# instead we run .ag({}) against the group by object and use a dictionary of column function pairs
result_1 = national_parks.groupby('region').agg({
    'visitors': ['mean', 'median'],
    'unit_name': ['nunique', 'count'],
    'year': ['min', 'max']
})

# this is an exampel where you can apply multiple functions to a single column 

print(result_1)

            visitors            unit_name        year      
                mean     median   nunique count   min   max
region                                                     
AK      1.204881e+05    24595.0         9   488  1922  2016
IM      7.931557e+05   378050.0        18  1590  1904  2016
MW      6.108661e+05   404700.0         7   533  1904  2016
NC      5.107866e+05   538297.0         1    46  1971  2016
NE      1.618013e+06  1699228.0         2   179  1919  2016
PW      6.627740e+05   407653.0        17  1393  1904  2016
SE      1.572087e+06   513397.0         7   453  1931  2016


In [92]:
print(type(result_1))

<class 'pandas.core.frame.DataFrame'>


In [96]:
survey.groupby('ph_class')['dissolved_oxygen_mg_L'].mean()

survey.groupby('ph_class').agg({
    'dissolved_oxygen_mg_L': 'mean'
})

,dissolved_oxygen_mg_L
ph_class,
acidic,6.350000
alkaline,9.948571
neutral,8.019181


In [100]:
# Write one .agg() call, grouped by region, that reports the median number of visitors, the latest year of record, 
# and the number of distinct states in each region. 
# Then say in one sentence why the median might be the more honest of the two averages for this data.

national_parks.groupby('region').agg({
    'visitors': 'median',
    'year': 'max',
    'state': 'nunique'

})

,visitors,year,state
region,,,
AK,24595.0,2016,1
IM,378050.0,2016,7
MW,404700.0,2016,6
NC,538297.0,2016,1
NE,1699228.0,2016,2
PW,407653.0,2016,6
SE,513397.0,2016,5


In [101]:
national_parks.groupby('unit_name')['visitors'].mean().sort_values(ascending = False).head(10)

unit_name
Great Smoky Mountains National Park    6.069152e+06
Grand Canyon National Park             2.096805e+06
Cuyahoga Valley National Park          2.093105e+06
Olympic National Park                  1.960219e+06
Rocky Mountain National Park           1.765457e+06
Yosemite National Park                 1.715356e+06
Grand Teton National Park              1.680664e+06
Acadia National Park                   1.668492e+06
Shenandoah National Park               1.556940e+06
Yellowstone National Park              1.549028e+06
Name: visitors, dtype: float64

In [102]:
park_stats = national_parks.groupby('unit_name')['visitors'].agg(['count', 'mean', 'max'])

park_stats.sort_values('mean', ascending = False).head(10)

,count,mean,max
unit_name,,,
Great Smoky Mountains National Park,86,6.069152e+06,11312786.0
Grand Canyon National Park,98,2.096805e+06,5969811.0
Cuyahoga Valley National Park,39,2.093105e+06,3527837.0
Olympic National Park,82,1.960219e+06,3846709.0
Rocky Mountain National Park,102,1.765457e+06,4517585.0
Yosemite National Park,111,1.715356e+06,5028868.0
Grand Teton National Park,88,1.680664e+06,3352500.0
Acadia National Park,98,1.668492e+06,5440952.0
Shenandoah National Park,81,1.556940e+06,2789100.0


In [116]:
(national_parks[national_parks['year'] == '2016']
.copy()
.groupby('state')['visitors']
.agg(['count', 'mean']).sort_values('mean')
)



,count,mean
state,,
MI,1,2.496600e+04
AS,1,2.889200e+04
SC,1,1.438430e+05
NV,1,1.448460e+05
MN,1,2.419120e+05
AK,9,2.450048e+05
TX,2,2.850645e+05
VI,1,4.113430e+05
NM,1,4.667730e+05


In [117]:
# 1.

filtered = national_parks[national_parks['year'] == '2016'].copy()

# 2.
grouped = filtered.groupby('state')

# 3.
result = grouped['visitors'].agg(['count', 'mean'])

#4. 
result.sort_values('mean')

,count,mean
state,,
MI,1,2.496600e+04
AS,1,2.889200e+04
SC,1,1.438430e+05
NV,1,1.448460e+05
MN,1,2.419120e+05
AK,9,2.450048e+05
TX,2,2.850645e+05
VI,1,4.113430e+05
NM,1,4.667730e+05


In [123]:
result_2 = national_parks.groupby(['region', 'state'])['visitors'].agg(['mean'])
print(result_2)

                      mean
region state              
AK     AK     1.204881e+05
IM     AZ     1.009639e+06
       CO     6.353273e+05
       MT     9.908449e+05
       NM     4.702549e+05
       TX     1.935455e+05
       UT     5.792425e+05
       WY     1.606659e+06
MW     AR     8.443021e+05
       MI     1.391706e+04
       MN     2.161420e+05
       ND     4.142048e+05
       OH     2.093105e+06
       SD     5.786315e+05
NC     VA     5.107866e+05
NE     ME     1.668492e+06
       VA     1.556940e+06
PW     AS     8.745692e+03
       CA     6.076323e+05
       HI     8.838688e+05
       NV     4.543101e+04
       OR     3.068096e+05
       WA     1.115853e+06
SE     FL     3.961249e+05
       KY     1.046299e+06
       NC     6.069152e+06
       SC     8.217781e+04
       VI     4.330022e+05


In [124]:
result_2.reset_index()

,region,state,mean
0,AK,AK,1.204881e+05
1,IM,AZ,1.009639e+06
2,IM,CO,6.353273e+05
3,IM,MT,9.908449e+05
4,IM,NM,4.702549e+05
5,IM,TX,1.935455e+05
6,IM,UT,5.792425e+05
7,IM,WY,1.606659e+06
8,MW,AR,8.443021e+05
9,MW,MI,1.391706e+04


In [125]:
pacific_west = national_parks[national_parks['region'] == 'PW']

pw_stats = pacific_west.groupby('unit_name')['visitors'].agg(['count', 'mean', 'max'])

pw_stats.sort_values('mean', ascending=False).head(10)

,count,mean,max
unit_name,,,
Olympic National Park,82,1.960219e+06,3846709.0
Yosemite National Park,111,1.715356e+06,5028868.0
Haleakala National Park,57,9.164551e+05,1963187.0
Hawai'i Volcanoes National Park,96,8.645207e+05,2247974.0
Mount Rainier National Park,113,8.361841e+05,1925100.0
Joshua Tree National Park,76,7.458512e+05,2505286.0
Sequoia National Park,111,5.662813e+05,1254688.0
Death Valley National Park,84,5.477119e+05,1296283.0
Kings Canyon National Park,113,4.688335e+05,1216800.0
